# 応用編12

V3.7新機能:ストレージ機能

このノートブックでは、スマートコントラクト・バージョン3.7から利用できるストレージ機能について、使い方を説明します。 


## ストレージ機能の概要

ストレージ機能では、クライアントから直接アクセスできるストレージ領域をブロックチェーン・ノード上に用意します。  
ストレージ領域に格納できるデータは任意のバイト列であり内容に制限はありません。各ノードごとに内容が異なっていても構いません。  
ブロックチェーン上のストレージコントラクト（スマートコントラクトの一種）を通して、ストレージ領域の作成や削除などの管理を行います。  
読み込みアクセスと書き込みアクセスのそれぞれについて、ストレージ領域へのアクセス権を設定できます。  
複数のストレージ領域が作成でき、ストレージ領域はストレージコントラクトIDとkey文字列によって識別されます。  
ひとつのストレージ領域は複数のチャンクに分けられ、チャンクは0から始まる通し番号(チャンク番号)で識別されます。  

## 準備

設定やライブラリを読み込みます。  

In [1]:
var { api } = await import('../lib/load-blockchain-api.mjs');
var { domain, cnfstr } = await import('../lib/load-config.mjs');
var { adminWallet, rpc } = await import('../lib/notebook-util.mjs');

ピア構成情報(cnfstr)から、ピア構成情報オブジェクト(peerscnf)を取得します。

In [2]:
var peerscnf = await api.loadPeersCnf(cnfstr);

ブロックチェーンに問い合わせ、peerscnfを最新の状態に更新します。

In [3]:
var peerscnf = await rpc.updatePeersCnf(peerscnf);

## ストレージ領域の作成

ストレージ領域を作成するには、先にストレージコントラクトが必要です。  
ストレージコントラクトの作成は、システムコントラクトc1createから行えます。  

In [4]:
var resp = await rpc.call(adminWallet, 'c1create', { type:'storage' });
var cid1 = resp.value;
console.log(resp);

{
  txno: 40392,
  txid: 'xiGSzfgfWouJKWArCHQCZwgZHfTyNTXFeNu5EuZY3xwdDB',
  status: 'ok',
  value: 'c343087979'
}


ストレージコントラクトは複数作ることができます。  

In [5]:
var resp = await rpc.call(adminWallet, 'c1create', { type:'storage' });
var cid2 = resp.value;
console.log(resp);

{
  txno: 40393,
  txid: 'xtB5sRbgRQC52rTSLwqSTwDPB7GJunxqGyLDSXkoMKAhHB',
  status: 'ok',
  value: 'c371784894'
}


なお、ストレージコントラクトは、ブロックチェーン上に保存されているため、毎回作成する必要はありません。  
(すでにストレージコントラクトを作成済みの場合は、それを使うことができます。)  　　

次に、ストレージ領域を作成します。  
ストレージ領域の作成は、ストレージコントラクトを通して行えます。  
この例では、ストレージコントラクトcid1を呼び出して作成します。

In [6]:
var key = 'test1'; // 作成するストレージ領域の名前（ストレージコントラクト内でユニークとする）
var expiry = Date.now() + 123000; // 現時点から123秒後まで有効
var resp = await rpc.call(adminWallet, cid1, { cmd:'create', key, expiry, chunksize:40000, chunks:100, readable:[domain], writable:[domain] });
console.log(resp);

{
  txno: 40394,
  txid: 'xdZGLNmu6Mf93dFJXZjn6zrZdk6RWQBM67Nh6h4k9YzqHB',
  status: 'ok',
  value: null
}


 作成時の引数の意味  
- key: 作成するストレージ領域を識別するための文字列  
- expiry: 有効期限(UNIXタイム)  
- chunksize: ひとつのチャンクのサイズ  
- chunks: チャンクの総数  
- readable: 読み出し権の付与先  
- writable: 書き込み権の付与先  

別の名前でストレージ領域を作成することができます。

In [7]:
var key = 'test2';
var resp = await rpc.call(adminWallet, cid1, { cmd:'create', key, expiry, chunksize:40000, chunks:20, readable:[domain], writable:[domain] });
console.log(resp);

{
  txno: 40395,
  txid: 'xyYiyjPgxyGC2vJmjqub66qgXycsMLmGyLBVMHXWxtb7u',
  status: 'ok',
  value: null
}


同じストレージコントラクト下で、同じ名前でストレージ領域を作成しようとすると、エラーになります。

In [8]:
var key = 'test2';
var resp = await rpc.call(adminWallet, cid1, { cmd:'create', key, expiry, chunksize:40000, chunks:20, readable:[domain], writable:[domain] });
console.log(resp);

{
  txno: 40396,
  txid: 'xTsze5jqeGUkcPYsNvQupVGuj64DRvJ68K8t6PMLiCRZw',
  status: 'denied',
  value: 'key reused'
}


同じ名前でも、ストレージコントラクトが異なれば違うストレージ領域とみなされ、作成できます。

In [9]:
var key = 'test2';
var resp = await rpc.call(adminWallet, cid2, { cmd:'create', key, expiry, chunksize:40000, chunks:20, readable:[domain], writable:[domain] });
console.log(resp);

{
  txno: 40397,
  txid: 'xcVotWETz8mn2fhDxQvYwpzwiENdq9f7TEtXRDEF4dK69',
  status: 'ok',
  value: null
}


## ストレージ領域への書き込み・読み出し

ストレージ領域へのデータの書き込みと読み出しを、クライアント側から行います。 

アクセス先のブロックチェーン・ノードの情報を取得します。  
ここでは、ピア構成情報に最初にリストされるピアのIDとURLを取得します。  

In [10]:
var pid = peerscnf.pids[0];
var { url } = peerscnf.peers.get(pid);
console.log(pid, url);

p00000000 https://tokyo.dncware-blockchain.com


ピアへ接続するためのソケットを作成します。  

In [11]:
var socket = api.createSocket(url);

ストレージ領域へ書き込むチャンクデータを作成します。

In [12]:
var chunkData = api.makeRandomUint8Array(40000); // 書き込むデータは何でもよい。ただし、chunksize以下であること。

下記の手順で、書き込みリクエストをピアに送信し、結果を取得します。  
・api.createRequest()でトランザクション要求を作成します。  
・api.signRequest()でトランザクション要求に署名します。  
・api.callStorage()でトランザクション要求と書き込みデータを送信します。  
トランザクションの作成・署名に使用したウォレットに紐づくユーザが、書き込みを行うユーザとなります。  

In [13]:
var request = await api.createRequest(adminWallet.address, cid1, { cmd:'write', key: 'test1', chunk: 0, peer: pid });
await api.signRequest(request, adminWallet, rpc.chainID);
var resp = await api.callStorage(socket, request, chunkData);
console.log(resp);

{ status: 'ok', ver: 1, pid: 'p00000000' }


次に、データを読み出します。

下記の手順で、読み出しリクエストをピアに送信し、結果を取得します。  
・api.createRequest()でトランザクション要求を作成します。  
・api.signRequest()でトランザクション要求に署名します。  
・api.callStorage()でトランザクション要求を送信します。  
トランザクションの作成・署名に使用したウォレットに紐づくユーザが、読み込みを行うユーザとなります。  

In [14]:
var request = await api.createRequest(adminWallet.address, cid1, { cmd:'read', key: 'test1', chunk: 0, peer: pid });
await api.signRequest(request, adminWallet, rpc.chainID);
var resp = await api.callStorage(socket, request);
console.log(resp);

{
  status: 'ok',
  ver: 1,
  pid: 'p00000000',
  data: <Buffer 9b 56 1e c4 e1 fd 73 70 85 e6 3b d1 3d af 99 bc 7f 8d ab 35 1b 0d 33 e2 c9 15 6c 2d c8 71 63 a4 f9 df 91 b1 75 66 35 e6 8a c3 d1 10 88 8e 33 2f 6f 56 ... 39950 more bytes>
}


書き込んだデータと読み出したデータが一致することを確認します。

In [15]:
console.log(Buffer.compare(resp.data, chunkData));

0


同じストレージ領域について、別のピアから読み出してみます。  

In [16]:
var pid = peerscnf.pids[1];
var { url } = peerscnf.peers.get(pid);
console.log(pid, url);

p14048874 https://canada.dncware-blockchain.com


In [17]:
var socket = api.createSocket(url);

In [18]:
var request = await api.createRequest(adminWallet.address, cid1, { cmd:'read', key: 'test1', chunk: 0, peer: pid });
await api.signRequest(request, adminWallet, rpc.chainID);
var resp = await api.callStorage(socket, request);
console.log(resp);

{ status: 'no data', pid: 'p14048874' }


ストレージ領域は、ピアごとにデータを格納するため、status:'no data'のエラーになります。

## ストレージ領域の削除

ストレージ領域は有効期限を過ぎると自動的に削除されますが、それ以前に削除することもできます。  
ストレージ領域を削除したい場合、ストレージコントラクトを通して削除を要求します。  

In [19]:
var resp = await rpc.call(adminWallet, cid1, { cmd:'delete', key: 'test1' });
console.log(resp);

{
  txno: 40398,
  txid: 'xbG9R5UQRQajGMkL5JxJBgjoTYaPZKzKVSDwQzM2LYGzn',
  status: 'ok',
  value: null
}


削除したストレージ領域にクライアントからアクセスしようとすると、status:'deleted'のエラーになります。

In [20]:
var request = await api.createRequest(adminWallet.address, cid1, { cmd:'read', key: 'test1', chunk: 0, peer: pid });
await api.signRequest(request, adminWallet, rpc.chainID);
var resp = await api.callStorage(socket, request);
console.log(resp);

{ status: 'deleted', pid: 'p14048874' }


## クリーンアップ

上で作成したストレージコントラクトは今後使用しないため、削除します。  
なお、ストレージコントラクトを削除すると、関連するストレージ領域もあわせて削除されます。

In [21]:
var resp = await rpc.call(adminWallet, 'c1terminate', { id: cid1 });
console.log(resp);
var resp = await rpc.call(adminWallet, 'c1terminate', { id: cid2 });
console.log(resp);

{
  txno: 40399,
  txid: 'xQv3ioKDpgvWAVXJNzUJvXVQ3sFkb5qYKVpXQpwKupkYu',
  status: 'ok',
  value: 'c343087979'
}
{
  txno: 40400,
  txid: 'xn4vH3Yr8BvkmqLUVEpesBot3ehAZGRNi5SNauzVf33ZBB',
  status: 'ok',
  value: 'c371784894'
}
